# Subtask 1 Colab/Local Pipeline

This notebook builds a local pipeline for Subtask 1 of the Explainable Suicide Risk Detection challenge.

It covers:

- Environment checks for Colab or local Jupyter
- Uploading `train.xlsx` and `leaderboard.xlsx` in Colab
- Excel loading
- Label cleanup
- Grouped validation split by `anon_user_id`
- TF-IDF baseline classifier
- Rule-based evidence extraction
- Optional transformer fine-tuning
- Submission CSV generation

If you only want to check model quality, run through the validation score cells and skip the final submission section for now.

Subtask 2 is skipped, so `factors` is always `[]`.

## 1. Environment Setup

Run this cell first. In Colab, use **Runtime > Change runtime type > T4 GPU** before running the optional transformer section.

This cell installs only missing packages. If it installs anything, continue with the next cell afterward.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "accelerate": "accelerate",
    "evaluate": "evaluate",
}

missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
print("Missing packages:", missing if missing else "none")

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("Installed missing packages. Continue with the next cell.")

Missing packages: ['evaluate']
Installed missing packages. Continue with the next cell.


## 2. Imports, Paths, and Reproducibility

In [2]:
from pathlib import Path
import ast
import json
import os
import random
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    HAS_STRATIFIED_GROUP_KFOLD = False

import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

LOCAL_PROJECT_ROOT = Path("/home/yibo/Desktop/Suicide")
if IN_COLAB:
    ROOT = Path("/content")
elif LOCAL_PROJECT_ROOT.exists():
    ROOT = LOCAL_PROJECT_ROOT
else:
    ROOT = Path.cwd()

os.chdir(ROOT)

TRAIN_PATH = ROOT / "train.xlsx"
LEADERBOARD_PATH = ROOT / "leaderboard.xlsx"
OUTPUT_DIR = ROOT / "outputs"
MODEL_DIR = ROOT / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

USE_TRANSFORMER_FOR_SUBMISSION = False

LABELS = ["Indicator", "Ideation", "Behavior", "Attempt"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

print("Running in Colab:", IN_COLAB)
print("Project directory:", ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Running in Colab: True
Project directory: /content
CUDA available: True
GPU: Tesla T4


## 3. Upload Data Files in Colab

If you are running in Colab, upload `train.xlsx` and `leaderboard.xlsx` when prompted. If the files are already in `/content`, this cell will skip uploading.

In [3]:
expected_files = ["train.xlsx", "leaderboard.xlsx"]

if IN_COLAB:
    missing_files = [name for name in expected_files if not (ROOT / name).exists()]
    if missing_files:
        print("Please upload:", missing_files)
        from google.colab import files
        uploaded = files.upload()
        print("Uploaded:", list(uploaded.keys()))
    else:
        print("Both Excel files already found in /content.")
else:
    print("Local mode. Looking for files in:", ROOT)

for name in expected_files:
    path = ROOT / name
    print(name, "FOUND" if path.exists() else "MISSING", path)

assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH}. Upload or place train.xlsx in {ROOT}."
assert LEADERBOARD_PATH.exists(), f"Missing {LEADERBOARD_PATH}. Upload or place leaderboard.xlsx in {ROOT}."

Please upload: ['leaderboard.xlsx']


Saving leaderboard.xlsx to leaderboard.xlsx
Uploaded: ['leaderboard.xlsx']
train.xlsx FOUND /content/train.xlsx
leaderboard.xlsx FOUND /content/leaderboard.xlsx


## 4. Load and Clean Data

In [4]:
def normalize_risk_label(value):
    text = str(value).strip().lower()
    mapping = {
        "indicator": "Indicator",
        "ideation": "Ideation",
        "behavior": "Behavior",
        "attempt": "Attempt",
    }
    if text not in mapping:
        raise ValueError(f"Unknown risk label: {value!r}")
    return mapping[text]


def clean_evidence(value):
    if pd.isna(value):
        return []
    spans = []
    seen = set()
    for span in str(value).split(";"):
        span = re.sub(r"\s+", " ", span).strip()
        key = span.lower()
        if span and key not in seen:
            spans.append(span)
            seen.add(key)
    return spans


train_raw = pd.read_excel(TRAIN_PATH)
leaderboard = pd.read_excel(LEADERBOARD_PATH)

train = train_raw.rename(columns={
    "suicide risk": "risk_level",
    "evidence for suicide risk level": "evidence",
}).copy()

train["risk_level"] = train["risk_level"].apply(normalize_risk_label)
train["evidence_spans"] = train["evidence"].apply(clean_evidence)
train["post"] = train["post"].fillna("").astype(str)
leaderboard["post"] = leaderboard["post"].fillna("").astype(str)

print("Train shape:", train.shape)
print("Leaderboard shape:", leaderboard.shape)
print("Class counts:")
display(train["risk_level"].value_counts().reindex(LABELS))
display(train.head(3))

Train shape: (1635, 8)
Leaderboard shape: (378, 4)
Class counts:


,count
risk_level,
Indicator,611
Ideation,519
Behavior,391
Attempt,114


,row_id,anon_user_id,post_id,post,risk_level,evidence,factors,evidence_spans
0,P00000,U0122,0,My thoughts are getting worse. Already am suff...,Ideation,just wanna die; just wanna fucking die,"['mental health issues', 'stressful life event...","[just wanna die, just wanna fucking die]"
1,P00001,U0122,1,I'm done with this healing and everything.. I'...,Ideation,going to die,"['hopelessness', 'hopelessness', 'coping strat...",[going to die]
2,P00002,U0122,2,Tired. 12:34am at night. 6% charge on my phone...,Ideation,pass away in my sleep; do not want to wake up;...,"['hopelessness', 'hopelessness']","[pass away in my sleep, do not want to wake up..."


## 5. Basic Data Checks

In [5]:
assert train["row_id"].isna().sum() == 0, "Missing row_id in train"
assert leaderboard["row_id"].isna().sum() == 0, "Missing row_id in leaderboard"
assert set(train["risk_level"].unique()) <= set(LABELS), "Invalid labels found"
assert leaderboard["row_id"].is_unique, "Leaderboard row_id values must be unique"

def span_in_post(span, post):
    return span.lower() in post.lower()

bad_evidence = []
for _, row in train.iterrows():
    for span in row["evidence_spans"]:
        if not span_in_post(span, row["post"]):
            bad_evidence.append((row["row_id"], span))

print("Rows:", len(train), "Leaderboard rows:", len(leaderboard))
print("Evidence spans not found verbatim ignoring case:", len(bad_evidence))
bad_evidence[:10]

Rows: 1635 Leaderboard rows: 378
Evidence spans not found verbatim ignoring case: 1199


[('P00033', 'none'),
 ('P00036', 'none'),
 ('P00039', 'none'),
 ('P00041', 'none'),
 ('P00047', 'none'),
 ('P00048', 'none'),
 ('P00050', 'none'),
 ('P00051', 'none'),
 ('P00054', 'none'),
 ('P00056', 'none')]

## 6. Grouped Validation Split

The split is grouped by `anon_user_id` so posts from the same user do not appear in both training and validation.

In [6]:
X = train["post"].values
y = train["risk_level"].values
groups = train["anon_user_id"].fillna("missing_user").astype(str).values

if HAS_STRATIFIED_GROUP_KFOLD:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_idx, valid_idx = next(splitter.split(X, y, groups))
else:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, valid_idx = next(splitter.split(X, y, groups))

train_df = train.iloc[train_idx].reset_index(drop=True)
valid_df = train.iloc[valid_idx].reset_index(drop=True)

overlap = set(train_df["anon_user_id"]) & set(valid_df["anon_user_id"])
assert not overlap, f"User leakage found: {len(overlap)} overlapping users"

print("Train split:", train_df.shape)
print("Valid split:", valid_df.shape)
print("Train label counts:")
display(train_df["risk_level"].value_counts().reindex(LABELS))
print("Valid label counts:")
display(valid_df["risk_level"].value_counts().reindex(LABELS))

Train split: (1305, 8)
Valid split: (330, 8)
Train label counts:


,count
risk_level,
Indicator,498
Ideation,402
Behavior,311
Attempt,94


Valid label counts:


,count
risk_level,
Indicator,113
Ideation,117
Behavior,80
Attempt,20


## 7. Baseline Risk Classifier

This baseline is fast and local. It gives us a working model before trying GPU fine-tuning.

In [7]:
baseline_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=60000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=2.0,
        solver="liblinear",
        random_state=SEED,
    )),
])

baseline_clf.fit(train_df["post"], train_df["risk_level"])
valid_pred = baseline_clf.predict(valid_df["post"])

weighted_f1 = f1_score(valid_df["risk_level"], valid_pred, average="weighted", labels=LABELS)
print("Validation weighted F1:", round(weighted_f1, 4))
print(classification_report(valid_df["risk_level"], valid_pred, labels=LABELS))
cm = pd.DataFrame(confusion_matrix(valid_df["risk_level"], valid_pred, labels=LABELS), index=LABELS, columns=LABELS)
display(cm)

Validation weighted F1: 0.5977
              precision    recall  f1-score   support

   Indicator       0.67      0.76      0.71       113
    Ideation       0.57      0.67      0.62       117
    Behavior       0.54      0.39      0.45        80
     Attempt       0.75      0.30      0.43        20

    accuracy                           0.61       330
   macro avg       0.63      0.53      0.55       330
weighted avg       0.61      0.61      0.60       330



,Indicator,Ideation,Behavior,Attempt
Indicator,86,23,3,1
Ideation,22,78,17,0
Behavior,16,32,31,1
Attempt,5,3,6,6


## 8. Evidence Extraction

This extractor returns short verbatim spans from the post. Tune the keyword lists and scoring after looking at validation errors.

In [8]:
RISK_PATTERNS = {
    "Ideation": [
        r"want(?:ed)? to die", r"wanna die", r"wish(?:ed)? i (?:was|were) dead",
        r"kill myself", r"end my life", r"suicidal thought", r"suicidal ideation",
        r"don'?t want to live", r"not want to live", r"never wake up", r"not wake up",
        r"better off dead", r"die peacefully", r"pass away",
    ],
    "Behavior": [
        r"have a plan", r"my plan", r"going to kill myself", r"i'?m going to die",
        r"tonight", r"tomorrow", r"overdose", r"pills?", r"rope", r"hang",
        r"jump", r"bridge", r"knife", r"cut", r"gun", r"note", r"goodbye",
    ],
    "Attempt": [
        r"attempted suicide", r"suicide attempt", r"tried to kill myself",
        r"tried killing myself", r"i overdosed", r"overdosed", r"survived",
        r"woke up in (?:the )?hospital", r"after my attempt", r"last attempt",
        r"when i tried", r"i hanged myself", r"cut too deep",
    ],
    "Indicator": [
        r"depress", r"hopeless", r"tired", r"worthless", r"useless", r"alone",
        r"anxious", r"therapy", r"crying", r"struggling",
    ],
}

ALL_PATTERNS = [(label, re.compile(pattern, flags=re.IGNORECASE)) for label, patterns in RISK_PATTERNS.items() for pattern in patterns]


def word_count(text):
    return len(re.findall(r"\w+", str(text)))


def split_candidates(post):
    post = str(post)
    candidates = []

    # Sentence-like chunks.
    for sent in re.split(r"(?<=[.!?])\s+|\n+", post):
        sent = sent.strip()
        if sent:
            candidates.append(sent)
            # Clause-like chunks inside long sentences.
            for clause in re.split(r"[,;:()\[\]{}]|\s+-\s+", sent):
                clause = clause.strip()
                if clause and clause != sent:
                    candidates.append(clause)

    # Exact regex matches expanded to a small phrase window.
    tokens = list(re.finditer(r"\S+", post))
    for _, pattern in ALL_PATTERNS:
        for match in pattern.finditer(post):
            token_positions = [i for i, tok in enumerate(tokens) if tok.start() <= match.end() and tok.end() >= match.start()]
            if not token_positions:
                candidates.append(match.group(0).strip())
                continue
            start_i = max(0, min(token_positions) - 3)
            end_i = min(len(tokens), max(token_positions) + 4)
            candidates.append(post[tokens[start_i].start():tokens[end_i - 1].end()].strip())

    # Deduplicate while preserving the original text as much as possible.
    deduped = []
    seen = set()
    for cand in candidates:
        # Do not normalize internal whitespace here. Evidence should remain a verbatim substring.
        cand = cand.strip(" \t\r\n")
        key = normalize_phrase(cand) if "normalize_phrase" in globals() else cand.lower().strip()
        if cand and key not in seen and 1 <= word_count(cand) <= 35:
            deduped.append(cand)
            seen.add(key)
    return deduped


def score_candidate(candidate, predicted_label):
    score = 0.0
    wc = word_count(candidate)
    if wc <= 8:
        score += 0.4
    elif wc <= 18:
        score += 0.2
    else:
        score -= 0.2

    for label, patterns in RISK_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, candidate, flags=re.IGNORECASE):
                score += 2.0 if label == predicted_label else 0.7

    # Attempt evidence is often explicit and should be boosted when predicted.
    if predicted_label == "Attempt" and re.search(r"attempt|tried|overdos|surviv|hospital", candidate, re.I):
        score += 1.2
    if predicted_label == "Behavior" and re.search(r"plan|going to|tonight|tomorrow|pills?|rope|hang|jump|bridge|knife|gun|goodbye", candidate, re.I):
        score += 1.0
    return score


def extract_evidence(post, predicted_label, max_spans=3):
    candidates = split_candidates(post)
    scored = [(score_candidate(c, predicted_label), c) for c in candidates]
    scored.sort(key=lambda x: x[0], reverse=True)

    selected = []
    selected_keys = set()
    threshold = 1.0 if predicted_label != "Indicator" else 1.4

    for score, cand in scored:
        key = cand.lower()
        if score < threshold:
            continue
        if any(key in old or old in key for old in selected_keys):
            continue
        selected.append(cand)
        selected_keys.add(key)
        if len(selected) >= max_spans:
            break

    return "; ".join(selected)


for i in range(3):
    print("Pred:", valid_pred[i])
    print("Evidence:", extract_evidence(valid_df.loc[i, "post"], valid_pred[i]))
    print("Gold:", valid_df.loc[i, "evidence_spans"])
    print("---")

Pred: Indicator
Evidence: 
Gold: ['none']
---
Pred: Ideation
Evidence: i want to kill myself because of it
Gold: ['want to kill myself']
---
Pred: Ideation
Evidence: 
Gold: ['going to die']
---


## 9. Approximate Phrase F1 for Evidence

In [9]:
def normalize_phrase(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def split_predicted_evidence(text):
    if pd.isna(text) or not str(text).strip():
        return []
    return [s.strip() for s in str(text).split(";") if s.strip()]


def phrase_f1_one(gold_spans, pred_spans):
    gold = [normalize_phrase(x) for x in gold_spans if normalize_phrase(x)]
    pred = [normalize_phrase(x) for x in pred_spans if normalize_phrase(x)]
    if not gold and not pred:
        return 1.0, 1.0, 1.0
    if not gold or not pred:
        return 0.0, 0.0, 0.0

    matched_gold = set()
    matched_pred = set()

    for pi, p in enumerate(pred):
        for gi, g in enumerate(gold):
            if gi in matched_gold:
                continue
            p_len = max(1, word_count(p))
            g_len = max(1, word_count(g))
            length_ok = p_len <= 3 * g_len
            overlap_ok = p in g or g in p
            if length_ok and overlap_ok:
                matched_pred.add(pi)
                matched_gold.add(gi)
                break

    precision = len(matched_pred) / len(pred) if pred else 0.0
    recall = len(matched_gold) / len(gold) if gold else 0.0
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return precision, recall, f1


valid_evidence_pred = [extract_evidence(post, label) for post, label in zip(valid_df["post"], valid_pred)]
phrase_scores = [phrase_f1_one(gold, split_predicted_evidence(pred)) for gold, pred in zip(valid_df["evidence_spans"], valid_evidence_pred)]

phrase_precision = np.mean([x[0] for x in phrase_scores])
phrase_recall = np.mean([x[1] for x in phrase_scores])
phrase_f1 = np.mean([x[2] for x in phrase_scores])

print("Approx phrase precision:", round(phrase_precision, 4))
print("Approx phrase recall:", round(phrase_recall, 4))
print("Approx phrase F1:", round(phrase_f1, 4))

valid_review = valid_df[["row_id", "post", "risk_level", "evidence_spans"]].copy()
valid_review["pred_risk"] = valid_pred
valid_review["pred_evidence"] = valid_evidence_pred
valid_review["phrase_f1"] = [x[2] for x in phrase_scores]
display(valid_review.sort_values("phrase_f1").head(10))

Approx phrase precision: 0.1025
Approx phrase recall: 0.1081
Approx phrase F1: 0.0992


,row_id,post,risk_level,evidence_spans,pred_risk,pred_evidence,phrase_f1
183,P01078,Why am I a worthless piece of garbage? I’m so ...,Ideation,"[feel like hurting mysef, want to die, don't w...",Ideation,I’m so hurt and lonely I want to die.; and lon...,0.0
191,P01172,me and my mom had a talk again i emailed my te...,Behavior,[wanna shoot myself],Ideation,teacher about my suicidal thoughts and i told;...,0.0
34,P00249,Escape No escape,Indicator,[none],Indicator,,0.0
189,P01170,i really don’t wanna live another day if i cou...,Behavior,[shoot myself in the back of the head right now],Behavior,,0.0
188,P01169,my grandpa has this thing full of guns maybe i...,Behavior,"[guns, used one]",Indicator,,0.0
187,P01168,is it worth it should i do it im considering i...,Ideation,[do it],Behavior,,0.0
35,P00250,Creep I record people during school hours. its...,Indicator,[none],Indicator,,0.0
185,P01080,Why am I a Worthless Piece of Shit? I’m so dep...,Ideation,"[feel like hurting mysef, want to die, don't w...",Ideation,I’m so hurt and lonely I want to die.; and lon...,0.0
184,P01079,Why am I a Worthless Piece of Shit? I’m so dep...,Ideation,"[feel like hurting mysef, want to die, don't w...",Ideation,I’m so hurt and lonely I want to die.; and lon...,0.0
288,P01875,it isn't fair i want to die. it isn't fair tha...,Ideation,[Main phrases that lead to this assessment: wa...,Ideation,it isn't fair i want to die.; isn't fair i wan...,0.0


## 10. Optional Transformer Fine-Tuning

The baseline above is fast. For a stronger model in Colab, set `RUN_TRANSFORMER = True` and run the next cell. This may download a model the first time. Use a GPU runtime.

In [14]:
RUN_TRANSFORMER = True
TRANSFORMER_MODEL_NAME = "roberta-base"  # Alternative: "microsoft/deberta-v3-base"

if RUN_TRANSFORMER:
    import inspect
    from datasets import Dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

    hf_train = train_df[["post", "risk_level"]].copy()
    hf_valid = valid_df[["post", "risk_level"]].copy()
    hf_train["label"] = hf_train["risk_level"].map(LABEL_TO_ID)
    hf_valid["label"] = hf_valid["risk_level"].map(LABEL_TO_ID)

    tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)

    def tokenize_batch(batch):
        return tokenizer(batch["post"], truncation=True, max_length=512)

    ds_train = Dataset.from_pandas(hf_train[["post", "label"]]).map(tokenize_batch, batched=True)
    ds_valid = Dataset.from_pandas(hf_valid[["post", "label"]]).map(tokenize_batch, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        TRANSFORMER_MODEL_NAME,
        num_labels=len(LABELS),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
    )

    class_counts = train_df["risk_level"].value_counts().reindex(LABELS).values.astype(float)
    class_weights = class_counts.sum() / (len(LABELS) * class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            weights = class_weights.to(outputs.logits.device)
            loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
            loss = loss_fn(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"weighted_f1": f1_score(labels, preds, average="weighted")}

    training_args_kwargs = dict(
        output_dir=str(MODEL_DIR / "risk_transformer"),
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="weighted_f1",
        greater_is_better=True,
        seed=SEED,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    # Transformers changed this argument name across versions.
    training_args_params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in training_args_params:
        training_args_kwargs["eval_strategy"] = "epoch"
    else:
        training_args_kwargs["evaluation_strategy"] = "epoch"

    args = TrainingArguments(**training_args_kwargs)
    trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_valid,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

    trainer_params = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = WeightedTrainer(**trainer_kwargs)

    trainer.train()
    def predict_with_transformer(posts, batch_size=16):
        pred_df = pd.DataFrame({"post": list(posts)})
        pred_ds = Dataset.from_pandas(pred_df).map(tokenize_batch, batched=True)
        logits = trainer.predict(pred_ds).predictions
        return [ID_TO_LABEL[int(i)] for i in np.argmax(logits, axis=-1)]

    transformer_logits = trainer.predict(ds_valid).predictions
    transformer_pred = [ID_TO_LABEL[int(i)] for i in np.argmax(transformer_logits, axis=-1)]
    print("Transformer weighted F1:", f1_score(valid_df["risk_level"], transformer_pred, average="weighted", labels=LABELS))
    print(classification_report(valid_df["risk_level"], transformer_pred, labels=LABELS))
    USE_TRANSFORMER_FOR_SUBMISSION = True
else:
    USE_TRANSFORMER_FOR_SUBMISSION = False
    print("Transformer fine-tuning skipped. Set RUN_TRANSFORMER = True to train in Colab with a GPU.")

Map:   0%|          | 0/1305 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Weighted F1
1,No log,1.003188,0.651514
2,No log,0.733960,0.733499
3,No log,0.806181,0.746066


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Transformer weighted F1: 0.746066369101334
              precision    recall  f1-score   support

   Indicator       0.87      0.78      0.82       113
    Ideation       0.76      0.74      0.75       117
    Behavior       0.63      0.72      0.67        80
     Attempt       0.54      0.65      0.59        20

    accuracy                           0.74       330
   macro avg       0.70      0.72      0.71       330
weighted avg       0.75      0.74      0.75       330



In [17]:
valid_evidence_pred_transformer = [
    extract_evidence(post, label)
    for post, label in zip(valid_df["post"], transformer_pred)
]

phrase_scores_transformer = [
    phrase_f1_one(gold, split_predicted_evidence(pred))
    for gold, pred in zip(valid_df["evidence_spans"], valid_evidence_pred_transformer)
]

phrase_precision = np.mean([x[0] for x in phrase_scores_transformer])
phrase_recall = np.mean([x[1] for x in phrase_scores_transformer])
phrase_f1 = np.mean([x[2] for x in phrase_scores_transformer])

print("Transformer evidence approx phrase precision:", round(phrase_precision, 4))
print("Transformer evidence approx phrase recall:", round(phrase_recall, 4))
print("Transformer evidence approx phrase F1:", round(phrase_f1, 4))

valid_review_transformer = valid_df[["row_id", "post", "risk_level", "evidence_spans"]].copy()
valid_review_transformer["pred_risk"] = transformer_pred
valid_review_transformer["pred_evidence"] = valid_evidence_pred_transformer
valid_review_transformer["phrase_f1"] = [x[2] for x in phrase_scores_transformer]

display(valid_review_transformer.sort_values("phrase_f1").head(20))

Transformer evidence approx phrase precision: 0.1157
Transformer evidence approx phrase recall: 0.1186
Transformer evidence approx phrase F1: 0.11


,row_id,post,risk_level,evidence_spans,pred_risk,pred_evidence,phrase_f1
209,P01236,i am so depressed and lonely and hopeless. i d...,Indicator,[none],Ideation,i am so depressed and lonely and hopeless.; an...,0.0
218,P01471,... ...,Indicator,[Main phrases that lead to this assessment: none],Indicator,,0.0
217,P01418,"Suicide is man's way of telling God, 'You can'...",Indicator,[Main phrases that lead to this assessment: none],Ideation,,0.0
216,P01417,LIFE IS HARD LIFE IS HARD LIFE IS HARD LIFE IS...,Indicator,[Main phrases that lead to this assessment: none],Indicator,,0.0
215,P01416,Why is so hard to live with pain,Indicator,[Main phrases that lead to this assessment: none],Indicator,,0.0
214,P01415,Why there has so much pain,Indicator,[Main phrases that lead to this assessment: none],Indicator,,0.0
213,P01414,"Sometimes you think you want to disappear, but...",Indicator,[Main phrases that lead to this assessment: none],Indicator,,0.0
212,P01413,"People are not addicted to alcohol or drugs, t...",Indicator,[Main phrases that lead to this assessment: none],Indicator,,0.0
211,P01412,"Sleep just Isn't sleep anymore, It's an escape.",Indicator,[Main phrases that lead to this assessment: none],Indicator,,0.0
210,P01411,". 1.I'm tired \n2.Sleep \n1.No, You don't unde...",Indicator,[Main phrases that lead to this assessment: none],Indicator,1.I'm tired,0.0


## 11. Final Leaderboard Predictions

This cell uses the fine-tuned transformer if `RUN_TRANSFORMER = True` was used successfully. Otherwise, it trains the fast TF-IDF baseline on all training data.

In [15]:
final_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=60000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=2.0,
        solver="liblinear",
        random_state=SEED,
    )),
])

if USE_TRANSFORMER_FOR_SUBMISSION and "predict_with_transformer" in globals():
    print("Using the fine-tuned transformer for leaderboard predictions.")
    leaderboard_pred = predict_with_transformer(leaderboard["post"])
else:
    print("Using the TF-IDF + Logistic Regression baseline for leaderboard predictions.")
    final_clf.fit(train["post"], train["risk_level"])
    leaderboard_pred = final_clf.predict(leaderboard["post"])
leaderboard_evidence = [extract_evidence(post, label) for post, label in zip(leaderboard["post"], leaderboard_pred)]

submission = pd.DataFrame({
    "row_id": leaderboard["row_id"],
    "risk_level": leaderboard_pred,
    "evidence": leaderboard_evidence,
    "factors": ["[]"] * len(leaderboard),
})

display(submission.head())
display(submission["risk_level"].value_counts().reindex(LABELS))

Using the fine-tuned transformer for leaderboard predictions.


Map:   0%|          | 0/378 [00:00<?, ? examples/s]

,row_id,risk_level,evidence,factors
0,P00008,Ideation,I want to kill myself because of her.; to othe...,[]
1,P00009,Ideation,I want to kill myself because of her.; to othe...,[]
2,P00010,Ideation,I want to kill myself because of her.; to othe...,[]
3,P00011,Ideation,I want to kill myself because of her.; to othe...,[]
4,P00012,Ideation,I want to kill myself because of her.; to othe...,[]


,count
risk_level,
Indicator,121
Ideation,110
Behavior,114
Attempt,33


## 12. Submission Validation, Save, and Download

In [16]:
def validate_submission(submission, leaderboard, strict_evidence=False):
    required_columns = ["row_id", "risk_level", "evidence", "factors"]
    assert list(submission.columns) == required_columns, "Wrong submission columns"
    assert len(submission) == len(leaderboard), "Wrong submission row count"
    assert submission["row_id"].is_unique, "Duplicate row_id in submission"
    assert set(submission["row_id"]) == set(leaderboard["row_id"]), "row_id mismatch"
    assert set(submission["risk_level"]) <= set(LABELS), "Invalid risk labels"
    assert (submission["factors"] == "[]").all(), "factors must be [] for Subtask 1 only"

    post_by_id = dict(zip(leaderboard["row_id"], leaderboard["post"]))
    bad_spans = []
    for _, row in submission.iterrows():
        post = post_by_id[row["row_id"]]
        for span in split_predicted_evidence(row["evidence"]):
            if span.lower() not in post.lower():
                bad_spans.append((row["row_id"], span))
    if bad_spans:
        print(f"Warning: {len(bad_spans)} evidence spans were not exact substrings. Showing first 5:")
        print(bad_spans[:5])
        if strict_evidence:
            raise AssertionError(f"Evidence spans not found in source posts: {bad_spans[:5]}")
    else:
        print("All evidence spans are exact substrings of their source posts.")
    return True


validate_submission(submission, leaderboard, strict_evidence=False)

TEAM_NAME = "YourTeamName"  # Change this before official submission.
submission_path = OUTPUT_DIR / f"{TEAM_NAME}.csv"
submission.to_csv(submission_path, index=False)

read_back = pd.read_csv(submission_path)
assert len(read_back) == len(submission)
print("Saved:", submission_path)
display(read_back.head())

if IN_COLAB:
    from google.colab import files
    files.download(str(submission_path))

All evidence spans are exact substrings of their source posts.
Saved: /content/outputs/YourTeamName.csv


,row_id,risk_level,evidence,factors
0,P00008,Ideation,I want to kill myself because of her.; to othe...,[]
1,P00009,Ideation,I want to kill myself because of her.; to othe...,[]
2,P00010,Ideation,I want to kill myself because of her.; to othe...,[]
3,P00011,Ideation,I want to kill myself because of her.; to othe...,[]
4,P00012,Ideation,I want to kill myself because of her.; to othe...,[]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Next Improvements

After you get the first CSV:

1. Inspect validation mistakes in `valid_review`.
2. Improve keyword patterns for `Behavior` and `Attempt`.
3. Turn on transformer fine-tuning with `RUN_TRANSFORMER = True`.
4. Compare transformer validation weighted F1 against the TF-IDF baseline.
5. Use the better classifier for final leaderboard predictions.
6. Tune evidence extraction for phrase F1.